In [1]:
%load_ext autoreload
%autoreload 2

In [23]:
import numpy as np
import qutip as qt
from scipy.linalg import expm

from tensor_networks_simulations.mps.models import HBond
from tensor_networks_simulations.general_tools.ED_tools import spin_operator

In [51]:
Lx=2
Ly = 2
lattice = np.empty((Lx,Ly), np.ndarray)

In [52]:
lattice

array([[None, None],
       [None, None]], dtype=object)

In [24]:
J = -1.0
λ = -0.5
h = -0.1

dt = 0.01
#Hb = HBond(Lx, Jxs=Jx, Jys=Jy, Jzs=Jz, Hxs=hx, Hzs=hz, mus=mu, d=2)
#sp, sm , sz = Hb.sp, Hb.sm, Hb.sz

sx_b, sy_b, sz_b, sp_b, sm_b = spin_operator(4)

f = sp_b[0]*sp_b[1]*sm_b[2]*sm_b[3]
fp = f + f.dag()
field = -h*(sx_b[0]+sx_b[1]+sx_b[2]+sx_b[3])

H_p = np.array(J*f - λ*fp**2 +field)
qt.commutator(fp, field)

Quantum object: dims = [[2, 2, 2, 2], [2, 2, 2, 2]], shape = (16, 16), type = oper, isherm = False
Qobj data =
[[ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  -0.1  0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  -0.1  0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.1  0.   0.   0.   0.1  0.   0.   0.   0.   0.1
   0.1  0. ]
 [ 0.   0.   0.  -0.1  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  -0.1  0.
   0.   0. ]
 [ 0.   0.   0.  -0.1  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.   

In [75]:
for i in range(Lx):
    for j in range(Ly):
        lattice[i,j] = np.zeros((2,1,1,1,1))
        lattice[i,j][0,0,0,0,0] = 1

In [88]:
lattice[0,0] = np.zeros((2, 5, 1, 1, 4))
lattice[0,1] = np.zeros((2, 2, 4, 1, 1))
lattice[1,0] = np.zeros((2, 1, 1, 5, 3))
lattice[1,1] = np.zeros((2, 1, 3, 2, 1))

In [89]:
for i in range(2):
    for j in range(2):
        print(lattice[i,j].shape)

(2, 5, 1, 1, 4)
(2, 2, 4, 1, 1)
(2, 1, 1, 5, 3)
(2, 1, 3, 2, 1)


In [91]:
b_u = np.tensordot(lattice[0,0], lattice[0,1], axes=(4, 2))  # (s0, a1, a2, a3, s'0, a'1, a'3, a'4)
b_d = np.tensordot(lattice[1,0], lattice[1,1], axes=(4, 2))  # (s0, a1, a2, a3, s'0, a'1, a'3, a'4)
b = np.tensordot(b_u, b_d, axes=([1,5],[3,6]))  # (s0-0, a0-1, a0-2, s1-3, a1-4, a1-5, s2-6, a2-7, a2-8, s3-9, a3-10, a3-11)

H_block = np.reshape(expm(-dt*H_p), [2]*8)


θ = np.tensordot(H_block, b, axes=([0,1,2,3], [0,3,6,9])) # (s0-0, s1-1, s2-2, s3-3, a0-4, a0-5, a1-6, a1-7, a2-8, a2-9, a3-10, a3-11)
θ.shape

θ = np.transpose(θ, (0,1,2, 4,5,6,7,8,9, 3, 10, 11))
θb = np.reshape(θ, (θ.shape[0]*θ.shape[1]*θ.shape[2]*θ.shape[3]*θ.shape[4]*θ.shape[5]*θ.shape[6]*θ.shape[7]*θ.shape[8], θ.shape[9]*θ.shape[10]*θ.shape[11]) )

X, Y, Z = np.linalg.svd(θb)
A4 = np.sqrt(Y)*Z
A4

array([[0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j]])

In [46]:
a = np.array([[1, 2], [3, 4]])
a

array([[1, 2],
       [3, 4]])

In [47]:
a[0,0]

1